# Road Segmentation from Orthophotos — Demo Notebook

This notebook demonstrates the inference and visualization part of a road segmentation pipeline based on aerial orthophotos and binary road masks.

The goal is to show how a trained U-Net model can be used to predict road masks from image patches and compare the predictions with ground-truth masks.

> Raw geospatial data and full training datasets are not included in this repository due to file size and licensing constraints.


## 1. Project overview

The full pipeline used in this project is:

1. Load orthophotos (`.tif`) and road vector data (`.gml`)
2. Convert geometries to a common CRS (`EPSG:25833`)
3. Rasterize road polygons into binary masks
4. Generate aligned image/mask patches
5. Filter empty patches to reduce class imbalance
6. Split data by tile to avoid spatial leakage
7. Train a U-Net segmentation model
8. Evaluate predictions using Dice, IoU, Precision and Recall

This notebook focuses on steps **8** and **visual inspection of predictions**.


## 2. Imports

In [ ]:
from pathlib import Path
import random
import numpy as np
import matplotlib.pyplot as plt
import cv2
import tensorflow as tf

from tensorflow.keras.models import load_model
from tensorflow.keras.utils import load_img, img_to_array
from tensorflow.keras import backend as K

print("TensorFlow version:", tf.__version__)

gpus = tf.config.list_physical_devices("GPU")
print("GPU available:", len(gpus) > 0)


## 3. Configuration

Update the paths below depending on where your data and trained model are located.

Expected folder structure:

```text
examples/
  images/
    tile_001_0_0.jpg
  masks/
    tile_001_0_0.png

models/
  roads_extraction_final.h5
```


In [ ]:
# Input size used during training
INPUT_SIZE = (512, 512)

# Paths
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

IMAGE_DIR = PROJECT_ROOT / "examples" / "images"
MASK_DIR = PROJECT_ROOT / "examples" / "masks"

MODEL_PATH = PROJECT_ROOT / "models" / "roads_extraction_final.h5"

# If you want to use your local training folders instead, uncomment and edit:
# IMAGE_DIR = Path("/path/to/images")
# MASK_DIR = Path("/path/to/masks")
# MODEL_PATH = Path("/path/to/roads_extraction_final.h5")

print("Image directory:", IMAGE_DIR)
print("Mask directory:", MASK_DIR)
print("Model path:", MODEL_PATH)


## 4. Custom loss and metrics

The model was trained with a custom BCE + Dice loss.  
When loading a Keras `.h5` model, custom functions must be defined again and passed through `custom_objects`.

If you only want to use `model.predict`, you can also load the model with `compile=False`.


In [ ]:
def dice_coef(y_true, y_pred, smooth=1e-6):
    y_true_f = K.flatten(tf.cast(y_true, tf.float32))
    y_pred_f = K.flatten(tf.cast(y_pred > 0.5, tf.float32))
    intersection = K.sum(y_true_f * y_pred_f)
    return (2.0 * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)


def iou_score(y_true, y_pred, smooth=1e-6):
    y_true_f = K.flatten(tf.cast(y_true, tf.float32))
    y_pred_f = K.flatten(tf.cast(y_pred > 0.5, tf.float32))
    intersection = K.sum(y_true_f * y_pred_f)
    union = K.sum(y_true_f) + K.sum(y_pred_f) - intersection
    return (intersection + smooth) / (union + smooth)


def dice_loss(y_true, y_pred, smooth=1e-6):
    y_true_f = K.flatten(tf.cast(y_true, tf.float32))
    y_pred_f = K.flatten(tf.cast(y_pred, tf.float32))
    intersection = K.sum(y_true_f * y_pred_f)
    dice = (2.0 * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)
    return 1.0 - dice


def bce_dice_loss(y_true, y_pred):
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    return bce + dice_loss(y_true, y_pred)


## 5. Load the trained model

In [ ]:
# Option A: load with custom objects, useful if you want to evaluate with the same loss/metrics
if MODEL_PATH.exists():
    model = load_model(
        MODEL_PATH,
        custom_objects={
            "bce_dice_loss": bce_dice_loss,
            "dice_loss": dice_loss,
            "dice_coef": dice_coef,
            "iou_score": iou_score,
        }
    )
    print("Model loaded successfully.")
else:
    model = None
    print("Model file not found. Please update MODEL_PATH.")


## 6. Match image and mask files

Images and masks are matched by their filename stem.

Example:

```text
33-2-461-214-22_0_2816.jpg
33-2-461-214-22_0_2816.png
```


In [ ]:
def match_image_mask_files(image_dir, mask_dir, image_extensions=(".jpg", ".jpeg", ".png"), mask_extensions=(".png",)):
    image_dir = Path(image_dir)
    mask_dir = Path(mask_dir)

    image_files = sorted([
        p for p in image_dir.iterdir()
        if p.is_file() and p.suffix.lower() in image_extensions
    ])

    mask_files = sorted([
        p for p in mask_dir.iterdir()
        if p.is_file() and p.suffix.lower() in mask_extensions
    ])

    image_map = {p.stem: p for p in image_files}
    mask_map = {p.stem: p for p in mask_files}

    common_stems = sorted(set(image_map) & set(mask_map))

    matched_images = [image_map[s] for s in common_stems]
    matched_masks = [mask_map[s] for s in common_stems]

    print("Images found:", len(image_files))
    print("Masks found:", len(mask_files))
    print("Matched pairs:", len(common_stems))

    return matched_images, matched_masks


if IMAGE_DIR.exists() and MASK_DIR.exists():
    image_paths, mask_paths = match_image_mask_files(IMAGE_DIR, MASK_DIR)
else:
    image_paths, mask_paths = [], []
    print("Image or mask directory not found. Please update the paths.")


## 7. Helper functions for loading and visualization

In [ ]:
def load_image_and_mask(image_path, mask_path, input_size=(512, 512)):
    image = load_img(image_path, target_size=input_size, color_mode="rgb")
    image = img_to_array(image) / 255.0

    mask = load_img(mask_path, target_size=input_size, color_mode="grayscale")
    mask = img_to_array(mask)
    mask = (mask > 0).astype(np.uint8)

    return image.astype(np.float32), mask


def make_overlay(image_uint8, mask_2d, alpha=0.35, color=(255, 255, 255)):
    mask_2d = (mask_2d > 0).astype(np.uint8)

    color_mask = np.zeros_like(image_uint8)
    color_mask[:, :] = color

    overlay = np.where(
        mask_2d[:, :, None] > 0,
        (1 - alpha) * image_uint8 + alpha * color_mask,
        image_uint8
    )

    return overlay.astype(np.uint8)


def show_image_mask_pair(image_path, mask_path, input_size=(512, 512)):
    image, mask = load_image_and_mask(image_path, mask_path, input_size)
    image_uint8 = (image * 255).astype(np.uint8)
    mask_2d = mask.squeeze()

    fig, axes = plt.subplots(1, 3, figsize=(13, 4))

    axes[0].imshow(image_uint8)
    axes[0].set_title("Image")
    axes[0].axis("off")

    axes[1].imshow(mask_2d, cmap="gray")
    axes[1].set_title("Ground truth")
    axes[1].axis("off")

    axes[2].imshow(make_overlay(image_uint8, mask_2d))
    axes[2].set_title("Overlay")
    axes[2].axis("off")

    plt.tight_layout()
    plt.show()


## 8. Visualize a few image/mask pairs

In [ ]:
if len(image_paths) > 0:
    n = min(3, len(image_paths))
    for i in range(n):
        show_image_mask_pair(image_paths[i], mask_paths[i], input_size=INPUT_SIZE)
else:
    print("No image/mask pairs available.")


## 9. Predict and visualize results

In [ ]:
def plot_predictions_from_files(
    model,
    image_paths,
    mask_paths,
    input_size=(512, 512),
    n_samples=10,
    seed=42,
    threshold=0.5,
    save_path=None
):
    if model is None:
        print("Model is not loaded.")
        return

    if len(image_paths) != len(mask_paths):
        raise ValueError("image_paths and mask_paths must have the same length")

    if len(image_paths) == 0:
        print("No image/mask pairs available.")
        return

    n_samples = min(n_samples, len(image_paths))

    random.seed(seed)
    indices = random.sample(range(len(image_paths)), n_samples)

    selected_images = [image_paths[i] for i in indices]
    selected_masks = [mask_paths[i] for i in indices]

    images = []
    gt_masks = []

    for img_path, mask_path in zip(selected_images, selected_masks):
        image, mask = load_image_and_mask(img_path, mask_path, input_size=input_size)
        images.append(image)
        gt_masks.append(mask)

    images = np.array(images, dtype=np.float32)
    gt_masks = np.array(gt_masks, dtype=np.uint8)

    preds = model.predict(images, verbose=1)
    preds = (preds > threshold).astype(np.uint8)

    fig, axes = plt.subplots(n_samples, 4, figsize=(15, 3.5 * n_samples))

    if n_samples == 1:
        axes = np.expand_dims(axes, axis=0)

    for i in range(n_samples):
        image_uint8 = (images[i] * 255).astype(np.uint8)

        gt_mask = gt_masks[i].squeeze()
        pred_mask = preds[i].squeeze()

        pred_overlay = make_overlay(image_uint8, pred_mask, alpha=0.35)

        axes[i, 0].imshow(image_uint8)
        axes[i, 0].set_title(f"Image\n{selected_images[i].name}")
        axes[i, 0].axis("off")

        axes[i, 1].imshow(gt_mask, cmap="gray")
        axes[i, 1].set_title("Ground truth")
        axes[i, 1].axis("off")

        axes[i, 2].imshow(pred_mask, cmap="gray")
        axes[i, 2].set_title("Prediction")
        axes[i, 2].axis("off")

        axes[i, 3].imshow(pred_overlay)
        axes[i, 3].set_title("Predicted overlay")
        axes[i, 3].axis("off")

    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, bbox_inches="tight", dpi=150)

    plt.show()


In [ ]:
plot_predictions_from_files(
    model=model,
    image_paths=image_paths,
    mask_paths=mask_paths,
    input_size=INPUT_SIZE,
    n_samples=10,
    seed=42,
    threshold=0.5,
    save_path=PROJECT_ROOT / "results" / "prediction_examples.png" if (PROJECT_ROOT / "results").exists() else None
)


## 10. Compute simple metrics on the demo set

This section computes basic pixel-level metrics on the available example dataset.

For the full experiment, metrics should be computed on the held-out test set split by tile.


In [ ]:
def compute_numpy_metrics(y_true, y_pred, eps=1e-7):
    y_true = (y_true > 0).astype(np.uint8)
    y_pred = (y_pred > 0).astype(np.uint8)

    intersection = np.logical_and(y_true, y_pred).sum()
    union = np.logical_or(y_true, y_pred).sum()

    pred_sum = y_pred.sum()
    true_sum = y_true.sum()

    iou = (intersection + eps) / (union + eps)
    dice = (2 * intersection + eps) / (pred_sum + true_sum + eps)
    precision = (intersection + eps) / (pred_sum + eps)
    recall = (intersection + eps) / (true_sum + eps)

    return {
        "iou": iou,
        "dice": dice,
        "precision": precision,
        "recall": recall,
    }


def evaluate_demo_predictions(model, image_paths, mask_paths, input_size=(512, 512), threshold=0.5):
    if model is None:
        print("Model is not loaded.")
        return None

    if len(image_paths) == 0:
        print("No data available.")
        return None

    images = []
    gt_masks = []

    for img_path, mask_path in zip(image_paths, mask_paths):
        image, mask = load_image_and_mask(img_path, mask_path, input_size=input_size)
        images.append(image)
        gt_masks.append(mask.squeeze())

    images = np.array(images, dtype=np.float32)
    gt_masks = np.array(gt_masks, dtype=np.uint8)

    preds = model.predict(images, verbose=1)
    preds = (preds.squeeze() > threshold).astype(np.uint8)

    metrics = compute_numpy_metrics(gt_masks, preds)

    for key, value in metrics.items():
        print(f"{key}: {value:.4f}")

    return metrics


demo_metrics = evaluate_demo_predictions(
    model=model,
    image_paths=image_paths,
    mask_paths=mask_paths,
    input_size=INPUT_SIZE,
    threshold=0.5
)


## 11. Interpretation

Example test-set results from the full experiment:

| Metric | Value |
|---|---:|
| Accuracy | 0.994 |
| IoU | 0.577 |
| Dice | 0.621 |
| Precision | 0.643 |
| Recall | 0.676 |

Accuracy is high because the dataset is strongly imbalanced: most pixels are background.  
For road segmentation, more informative metrics are:

- **Dice coefficient**: overlap quality between predicted and true road masks
- **IoU**: stricter overlap metric
- **Precision**: how many predicted road pixels are actually roads
- **Recall**: how many true road pixels were detected

In this project, recall is especially important because missing road segments can strongly affect downstream surface-area estimation.


## 12. Notes and limitations

- The model is trained on patches, not full orthophotos.
- The split is performed by tile to reduce spatial data leakage.
- Raw orthophotos and GML files are not included due to size and licensing.
- Visual inspection remains important because vector-derived labels may not perfectly match the visual road surface.
- Future improvements could include stronger data augmentation, Tversky/Focal loss, and evaluation on geographically distinct test areas.
